# 06 — Model Serving endpoint

Deploys the same `@champion` model behind a low-latency REST endpoint with:

- **Scale-to-zero** (`scale_to_zero_enabled=True`) — costs nothing when idle
- **Small CPU** workload size — fine for a 25-feature tabular classifier
- **Inference table** auto-logging — every prediction lands in a Delta table the
  Lakehouse Monitor in `07` watches

Then we hit the endpoint with a sample shot and read the response.

In [1]:
import os, time, json
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput,
    AiGatewayConfig, AiGatewayInferenceTableConfig,
)

w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.xg_model"
ENDPOINT   = os.getenv("SERVING_ENDPOINT_NAME", "hockey-xg-endpoint")

# Find the champion version
from mlflow.tracking import MlflowClient
import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"Deploying {MODEL_NAME} v{champ.version} to endpoint '{ENDPOINT}'")

Deploying alexander_booth.hockey_xg_mlflow.xg_model v5 to endpoint 'hockey-xg-endpoint'


## Create (or update) the endpoint

This call is idempotent: if `ENDPOINT` already exists we update its config,
otherwise we create it. First-time creation takes ~3-5 minutes to provision.

In [2]:
served = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(champ.version),
    name="xg-champion",
    workload_size="Small",
    scale_to_zero_enabled=True,
)
inference_table = AiGatewayInferenceTableConfig(
    enabled=True,
    catalog_name=UC_CATALOG,
    schema_name=UC_SCHEMA,
    table_name_prefix="xg_endpoint",
)
ai_gateway = AiGatewayConfig(inference_table_config=inference_table)
cfg = EndpointCoreConfigInput(
    name=ENDPOINT,
    served_entities=[served],
)

existing = None
for ep in w.serving_endpoints.list():
    if ep.name == ENDPOINT:
        existing = ep
        break

if existing:
    print(f"Endpoint exists — updating to v{champ.version}")
    w.serving_endpoints.update_config(
        name=ENDPOINT,
        served_entities=cfg.served_entities,
    )
    w.serving_endpoints.put_ai_gateway(name=ENDPOINT, inference_table_config=inference_table)
else:
    print(f"Creating endpoint '{ENDPOINT}'")
    w.serving_endpoints.create(name=ENDPOINT, config=cfg, ai_gateway=ai_gateway)

print("Waiting for endpoint to become READY (this can take a few minutes on cold create)...")

Endpoint exists — updating to v5


Waiting for endpoint to become READY (this can take a few minutes on cold create)...


In [3]:
# Poll for readiness
for i in range(60):
    ep = w.serving_endpoints.get(ENDPOINT)
    state = ep.state.config_update if ep.state else None
    ready = ep.state.ready if ep.state else None
    print(f"  [{i:02d}] config_update={state}  ready={ready}")
    if ready and str(ready).endswith("READY") and (state is None or str(state).endswith("NOT_UPDATING")):
        break
    time.sleep(10)
print("Endpoint state:")
print(f"  ready: {ep.state.ready}")
print(f"  config_update: {ep.state.config_update}")

  [00] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [01] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [02] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [03] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [04] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


...

  [47] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [48] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [49] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [50] config_update=EndpointStateConfigUpdate.IN_PROGRESS  ready=EndpointStateReady.READY


  [51] config_update=EndpointStateConfigUpdate.NOT_UPDATING  ready=EndpointStateReady.READY
Endpoint state:
  ready: EndpointStateReady.READY
  config_update: EndpointStateConfigUpdate.NOT_UPDATING


## Query the endpoint with a sample shot

Two ways to query: the SDK convenience method, and the raw REST URL. We show
both so you can talk about however the customer's app would call it.

In [4]:
FEATURE_COLS = [
    "distance_ft", "angle_deg", "distance_sq", "angle_sq",
    "is_high_danger", "rebound", "rush", "rebound_dist", "rush_dist",
    "period", "time_in_period_sec", "prev_event_sec",
    "st_wrist", "st_snap", "st_slap", "st_backhand", "st_tip", "st_wrap",
    "str_5v5", "str_5v4_pp", "str_4v5_pk", "str_4v4", "str_3v3_ot", "str_6v5_en", "str_5v6",
]

# A high-danger rebound from the slot, even strength
sample_shot = {
    "distance_ft": 12.0, "angle_deg": 8.0,
    "distance_sq": 144.0, "angle_sq": 64.0,
    "is_high_danger": 1, "rebound": 1, "rush": 0,
    "rebound_dist": 12.0, "rush_dist": 0.0,
    "period": 2, "time_in_period_sec": 600, "prev_event_sec": 1.2,
    "st_wrist": 0, "st_snap": 1, "st_slap": 0, "st_backhand": 0, "st_tip": 0, "st_wrap": 0,
    "str_5v5": 1, "str_5v4_pp": 0, "str_4v5_pk": 0, "str_4v4": 0,
    "str_3v3_ot": 0, "str_6v5_en": 0, "str_5v6": 0,
}
# A low-danger long-distance slap shot from the point
weak_shot = {**sample_shot,
              "distance_ft": 55.0, "angle_deg": 25.0, "distance_sq": 3025.0,
              "angle_sq": 625.0, "is_high_danger": 0, "rebound": 0, "rebound_dist": 0.0,
              "st_snap": 0, "st_slap": 1}

batch = pd.DataFrame([sample_shot, weak_shot])
resp = w.serving_endpoints.query(name=ENDPOINT, dataframe_records=batch.to_dict(orient="records"))
print("Endpoint response:")
print(resp.as_dict())

Endpoint response:
{'predictions': [0.3135644724441374, 0.017591171245990578], 'served-model-name': 'xg-champion'}


## Inference table

`AutoCaptureConfig` writes every request/response pair (with timestamps and the
served model version) to a Delta table. We confirm it exists — the monitor in
`07` will use it as the data source.

In [5]:
auto_capture_table = f"{UC_CATALOG}.{UC_SCHEMA}.xg_endpoint_payload"
print(f"Inference table: {auto_capture_table}")
try:
    try:
        spark
    except NameError:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.serverless().getOrCreate()
    cnt = spark.table(auto_capture_table).count()
    print(f"  rows captured so far: {cnt}")
except Exception as e:
    print(f"  table not yet visible — it may take 1-2 min after first request to appear.")
    print(f"  error: {e}")

Inference table: alexander_booth.hockey_xg_mlflow.xg_endpoint_payload


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


  rows captured so far: 1


**Demo tip:** in the Serving UI for this endpoint, the *Use* tab shows a
curl snippet your customer's app would call. Copy/paste it during the live demo
for credibility.